# 01 · Muon S2 signal walk-through

            This notebook reproduces the per-channel **muon S2** characterisation that
            used to live in `sim_script/Muon_signal_sim.ipynb`. We

            1. load a small muon-track sample,
            2. interpolate dense energy-deposition points along each track,
            3. build the **expected S2 light pattern** for the top PMT array,
            4. compute the per-channel pe rate and mean charge.

            All paths come from `configs/muon_only.yaml`; edit that file to point at your
            local detector data before running.

In [ ]:
# Auto-discover the package even if the notebook is launched from outside the repo.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

In [ ]:
from relics_de_sim import DESimConfig, Pipeline
            from relics_de_sim.muon import compute_dense_muon_points

            cfg = DESimConfig.from_yaml('configs/muon_only.yaml')
            print('muon_track_dir:', cfg.paths.muon_track_dir)
            print('output_dir:', cfg.paths.output_dir)

## 1. Load a small muon batch

            We load the first few `muon_track.<i>.npy` files. Adjust ``files_per_batch``
            to scale the simulation horizon.

In [ ]:
from pathlib import Path
            files_per_batch = 2
            muon_files = [
                Path(cfg.paths.muon_track_dir) / f'muon_track.{i}.npy'
                for i in range(files_per_batch)
            ]
            pipe = Pipeline(cfg, rng=np.random.default_rng(0))
            pipe.load_muon_tracks(muon_files)
            print(f'n muons: {len(pipe.event_time)}')
            print(f'simulated wall-clock: {pipe.time_range_s:.2f} s')
            print(f'dense muon points:    {pipe.dense_muon.shape[0]}')

## 2. Inspect a single muon track

            The dense interpolation is the input to every downstream stage.

In [ ]:
ev_id = 0
            mask = pipe.dense_muon['eventId'] == ev_id
            pts = pipe.dense_muon[mask]
            fig, ax = plt.subplots(figsize=(6, 5))
            ax.scatter(pts['xd'], pts['yd'], c=pts['zd'], cmap='viridis', s=4)
            ax.set_aspect('equal')
            ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
            ax.set_title(f'Muon track {ev_id} (color = z [mm])')
            plt.colorbar(ax.collections[0], ax=ax, label='z [mm]')
            plt.show()

## 3. Per-event expected light pattern

            The `PatternSimulator` exposes the LCE × SE-gain product directly.  We use
            it here without applying the pile-up grouping or the area window.

In [ ]:
from relics_de_sim.pattern import PatternSimulator
            from relics_de_sim.lce import LCEMap

            xy = np.stack([pts['xd'], pts['yd']], axis=1)
            light_per_e = pipe.pattern_truth.per_electron_light(xy)
            print('light_per_e shape:', light_per_e.shape)

            channel_mean = light_per_e.mean(axis=0)
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.bar(np.arange(64), channel_mean[:64], label='top')
            ax.bar(np.arange(64, 128), channel_mean[64:], color='C3', label='bottom')
            ax.set_xlabel('PMT channel'); ax.set_ylabel('<pe / electron>')
            ax.set_title('Mean expected pe per channel for one muon track')
            ax.legend()
            plt.show()

## 4. Save metadata

            Everything we computed lives in memory; we typically persist the per-event
            light pattern alongside the muon summary.  See `scripts/run_de_sim.py` for
            the canonical save path.